# MLP Training and Feature Ablations

Train shallow 64-unit MLP and feature-group ablations. Standardization is fit on the training period only; class-balanced sample weights address label imbalance.

**Feature-set ablations:** full features; base features only; full features without balance features; without initiator activity features; and without recipient-risk features.


The six-step delay is the validation period. It is used to choose the fraud threshold, and is not used to fit model parameters. MLP early stopping also reserves a random portion of the training rows internally; this is not an additional chronological split.


## 1. Imports and paths

In [1]:
import os
import joblib
import pandas as pd
from sklearn.neural_network import MLPClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.utils.class_weight import compute_sample_weight

DATA_DIR = r"../results/featured_dataset/featured_144days.pkl"
RESULTS_DIR = r"../results/mlp"
RANDOM_STATE = 42


## 2. Shared chronological split and ablation groups

In [2]:
# Sort transactions by time before making chronological splits.
df = pd.read_pickle(DATA_DIR)
df = df.sort_values(["step", "index"]).reset_index(drop=True)

max_step = int(df["step"].max())
train_end = int(max_step * 0.8)
delay = 6
val_end = train_end + delay
test_start = val_end + 1

# Use all rows before the delay to train; use the delay period as validation.
train_df = df[df["step"] <= train_end]
val_df = df[(df["step"] > train_end) & (df["step"] <= val_end)]
test_df = df[df["step"] >= test_start]

print("Train steps:", train_df["step"].min(), "to", train_df["step"].max())
print("Validation (delay) steps:", val_df["step"].min(), "to", val_df["step"].max())
print("Test steps:", test_df["step"].min(), "to", test_df["step"].max())
print("Train rows:", len(train_df), " | Validation rows:", len(val_df), " | Test rows:", len(test_df))

# Separate labels from inputs and remove IDs and time from model features.
y_train = train_df["isFraud"]
y_val = val_df["isFraud"]
y_test = test_df["isFraud"]
drop_cols = ["step", "recipient", "initiator", "index", "isFraud"]
x_train = train_df.drop(columns=drop_cols)
x_val = val_df.drop(columns=drop_cols)
x_test = test_df.drop(columns=drop_cols)

# Group engineered features for the ablation experiments.
balance_features = [
    "amount_balance_ratio", "balance_increase_ratio", "neg_balance_i",
    "balance_error_i", "balance_error_r",
]
initiator_features = [
    "INIT_AVG_AMOUNT_TX_6", "INIT_AMOUNT_DEV_TX_6",
    "INIT_AVG_AMOUNT_TX_12", "INIT_AMOUNT_DEV_TX_12",
    "INIT_AVG_AMOUNT_TX_24", "INIT_AMOUNT_DEV_TX_24",
    "INIT_TX_COUNT_STEP_6", "INIT_TX_COUNT_STEP_12", "INIT_TX_COUNT_STEP_24",
]
recipient_features = [
    "RECIP_RISK_6", "RECIP_TX_COUNT_6", "RECIP_RISK_12",
    "RECIP_TX_COUNT_12", "RECIP_RISK_24", "RECIP_TX_COUNT_24",
]
engineered_features = balance_features + initiator_features + recipient_features
base_features = [col for col in x_train.columns if col not in engineered_features]

feature_sets = {
    "full": list(x_train.columns),
    "base_only": base_features,
    "without_balance": [col for col in x_train.columns if col not in balance_features],
    "without_initiator_activity": [col for col in x_train.columns if col not in initiator_features],
    "without_recipient_risk": [col for col in x_train.columns if col not in recipient_features],
}

print("Feature counts:")
for name, columns in feature_sets.items():
    print(name, len(columns))


Train steps: 0 to 114
Validation (delay) steps: 115 to 120
Test steps: 121 to 143
Train rows: 1361460  | Validation rows: 55663  | Test rows: 303058
Feature counts:
full 30
base_only 10
without_balance 25
without_initiator_activity 21
without_recipient_risk 24


# Train all ablations and save prediction artifacts

## 3. Hyperparameter ablation: alpha

Vary only L2 regularization strength (`alpha`) while keeping the full feature set, chronological split, training-only scaling, class-balanced sample weights, and all other model settings fixed. Compare validation average precision (AP); use validation results for model/threshold selection and reserve the test period for final evaluation.

In [ ]:
from sklearn.metrics import average_precision_score

# Keep the complete feature set fixed and fit scaling on training rows only.
full_features = list(x_train.columns)
scaler = StandardScaler()
x_train_full_scaled = scaler.fit_transform(x_train[full_features])
x_val_full_scaled = scaler.transform(x_val[full_features])
sample_weights = compute_sample_weight("balanced", y_train)

hyperparameter_values = [0.00001, 0.0001, 0.001, 0.01]
hyperparameter_results = []
hyperparameter_results_dir = os.path.join(RESULTS_DIR, "hyperparameter_ablation")
os.makedirs(hyperparameter_results_dir, exist_ok=True)

for alpha_value in hyperparameter_values:
    model = MLPClassifier(
        hidden_layer_sizes=(64,),
        activation="relu",
        solver="adam",
        alpha=alpha_value,
        batch_size="auto",
        learning_rate_init=0.001,
        max_iter=100,
        random_state=RANDOM_STATE,
        early_stopping=True,
        validation_fraction=0.1,
        n_iter_no_change=10,
    )
    model.fit(x_train_full_scaled, y_train, sample_weight=sample_weights)

    val_scores = model.predict_proba(x_val_full_scaled)[:, 1]
    artifact_name = f"alpha_{alpha_value:g}"

    joblib.dump(model, os.path.join(hyperparameter_results_dir, f"model_{artifact_name}.pkl"))
    joblib.dump(scaler, os.path.join(hyperparameter_results_dir, f"scaler_{artifact_name}.pkl"))
    pd.DataFrame({"isFraud": y_val, "score": val_scores}).to_pickle(
        os.path.join(hyperparameter_results_dir, f"{artifact_name}_validation.pkl")
    )
    hyperparameter_results.append({
        "setting": artifact_name,
        "alpha": alpha_value,
        "validation_average_precision": average_precision_score(y_val, val_scores),
        "training_iterations": model.n_iter_,
    })

results_df = pd.DataFrame(hyperparameter_results).sort_values(
    "validation_average_precision", ascending=False
)
print(results_df.drop(columns="setting").to_string(index=False))

# Select on validation only, then generate test predictions once for the selected setting.
best_result = max(hyperparameter_results, key=lambda result: result["validation_average_precision"])
selected_setting = best_result["setting"]
selected_model = joblib.load(os.path.join(hyperparameter_results_dir, f"model_{selected_setting}.pkl"))
selected_scaler = joblib.load(os.path.join(hyperparameter_results_dir, f"scaler_{selected_setting}.pkl"))
x_test_full_scaled = selected_scaler.transform(x_test[full_features])
test_scores = selected_model.predict_proba(x_test_full_scaled)[:, 1]
pd.read_pickle(os.path.join(hyperparameter_results_dir, f"{selected_setting}_validation.pkl")).to_pickle(
    os.path.join(hyperparameter_results_dir, "selected_validation.pkl")
)
pd.DataFrame({"isFraud": y_test, "score": test_scores}).to_pickle(
    os.path.join(hyperparameter_results_dir, "selected_test.pkl")
)
print(f"Validation-selected alpha: {best_result['alpha']}")

## 4. Hyperparameter ablation: hidden-layer width

Vary the number of neurons in a single hidden layer (`hidden_layer_sizes`) while keeping the full feature set, one-layer architecture, and all other MLP settings fixed. Compare validation average precision (AP), select the width using validation only, and reserve the test set for that selected configuration.

In [ ]:
from sklearn.metrics import average_precision_score

# Keep the complete feature set and one hidden layer fixed; vary only its width.
full_features = list(x_train.columns)
scaler = StandardScaler()
x_train_full_scaled = scaler.fit_transform(x_train[full_features])
x_val_full_scaled = scaler.transform(x_val[full_features])
sample_weights = compute_sample_weight("balanced", y_train)
hidden_layer_sizes_values = [(32,), (64,), (128,), (64, 32), (128, 64, 32)]
hidden_layer_sizes_results = []
hidden_layer_sizes_results_dir = os.path.join(RESULTS_DIR, "hyperparameter_ablation_hidden_layer_sizes")
os.makedirs(hidden_layer_sizes_results_dir, exist_ok=True)

for layer_sizes in hidden_layer_sizes_values:
    architecture = "-".join(map(str, layer_sizes))
    artifact_name = f"hidden_layer_sizes_{architecture}"
    
    model = MLPClassifier(
        hidden_layer_sizes=layer_sizes,
        activation="relu",
        solver="adam",
        alpha=0.0001,
        batch_size="auto",
        learning_rate_init=0.001,
        max_iter=100,
        random_state=RANDOM_STATE,
        early_stopping=True,
        validation_fraction=0.1,
        n_iter_no_change=10,
    )
    model.fit(x_train_full_scaled, y_train, sample_weight=sample_weights)

    val_scores = model.predict_proba(x_val_full_scaled)[:, 1]
    joblib.dump(model, os.path.join(hidden_layer_sizes_results_dir, f"model_{artifact_name}.pkl"))
    joblib.dump(scaler, os.path.join(hidden_layer_sizes_results_dir, f"scaler_{artifact_name}.pkl"))
    pd.DataFrame({"isFraud": y_val, "score": val_scores}).to_pickle(
        os.path.join(hidden_layer_sizes_results_dir, f"{artifact_name}_validation.pkl")
    )
    hidden_layer_sizes_results.append({
        "setting": artifact_name,
        "architecture": architecture,
        "n_layers": len(layer_sizes),
        "validation_average_precision": average_precision_score(y_val, val_scores),
        "training_iterations": model.n_iter_,
    })

print(pd.DataFrame(hidden_layer_sizes_results).drop(columns="setting").sort_values(
    "validation_average_precision", ascending=False
).to_string(index=False))

# Select using validation AP, then generate test predictions only for the selected model.
best_result = max(
    hidden_layer_sizes_results,
    key=lambda result: result["validation_average_precision"],
)
selected_setting = best_result["setting"]
selected_model = joblib.load(os.path.join(hidden_layer_sizes_results_dir, f"model_{selected_setting}.pkl"))
selected_scaler = joblib.load(os.path.join(hidden_layer_sizes_results_dir, f"scaler_{selected_setting}.pkl"))
x_test_full_scaled = selected_scaler.transform(x_test[full_features])
test_scores = selected_model.predict_proba(x_test_full_scaled)[:, 1]
pd.read_pickle(os.path.join(hidden_layer_sizes_results_dir, f"{selected_setting}_validation.pkl")).to_pickle(
    os.path.join(hidden_layer_sizes_results_dir, "selected_validation.pkl")
)
pd.DataFrame({"isFraud": y_test, "score": test_scores}).to_pickle(
    os.path.join(hidden_layer_sizes_results_dir, "selected_test.pkl")
)
print(f"Validation-selected hidden_layer_sizes: {selected_model.hidden_layer_sizes}")

 hidden_layer_size  validation_average_precision  training_iterations
               128                      0.350006                   14
                64                      0.349348                   13
                32                      0.348735                   21
                64                      0.348171                   13
               128                      0.348168                   13
Validation-selected hidden_layer_sizes: (128, 64, 32)


## 5. Train model with best parameters + feature ablation

In [ ]:
# # Create the results folder if it does not exist yet.
# os.makedirs(RESULTS_DIR, exist_ok=True)

# for feature_set_name, selected_features in feature_sets.items():
#     print("\nTraining MLP with:", feature_set_name)

#     # Fit scaling on the training data only, then apply it to validation and test data.
#     scaler = StandardScaler()
#     x_train_scaled = scaler.fit_transform(x_train[selected_features])
#     x_val_scaled = scaler.transform(x_val[selected_features])
#     x_test_scaled = scaler.transform(x_test[selected_features])

#     model = MLPClassifier(
#         hidden_layer_sizes=(64,),
#         activation="relu",
#         solver="adam",
#         alpha=0.0001,
#         batch_size="auto",
#         learning_rate_init=0.001,
#         max_iter=100,
#         random_state=RANDOM_STATE,
#         early_stopping=True,
#         validation_fraction=0.1,
#         n_iter_no_change=10,
#     )
#     sample_weights = compute_sample_weight("balanced", y_train)
#     print(model)
#     model.fit(x_train_scaled, y_train, sample_weight=sample_weights)

#     val_scores = model.predict_proba(x_val_scaled)[:, 1]
#     test_scores = model.predict_proba(x_test_scaled)[:, 1]

#     # Save the model, scaler, and scores for the evaluation notebook.
#     joblib.dump(model, os.path.join(RESULTS_DIR, f"model_{feature_set_name}.pkl"))
#     joblib.dump(scaler, os.path.join(RESULTS_DIR, f"scaler_{feature_set_name}.pkl"))
#     pd.DataFrame({"isFraud": y_val, "score": val_scores}).to_pickle(
#         os.path.join(RESULTS_DIR, f"{feature_set_name}_validation.pkl")
#     )
#     pd.DataFrame({"isFraud": y_test, "score": test_scores}).to_pickle(
#         os.path.join(RESULTS_DIR, f"{feature_set_name}_test.pkl")
#     )

#     print("Training iterations:", model.n_iter_)
#     print("Saved model and validation/test scores.")
